# 02 · Train the vetting model

Train a multiclass **XGBoost** classifier on the KOI features, evaluate it, compute a
SHAP explainer, and export `backend/models/vetting_xgb.joblib`. The app loads this at
startup; physics diagnostics override it when they conflict (see `exoseer_core.vet`).

This mirrors `ml/train_vetting_model.py`, which can also be run as a script.


In [ ]:
import sys, numpy as np, joblib
sys.path.insert(0, '../backend')
from app.exoseer_core.vet import MODEL_FEATURES
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
X, y = np.load('X.npy'), np.load('y.npy', allow_pickle=True)


### Encode labels, split, weight for class imbalance


In [ ]:
le = LabelEncoder().fit(y); ye = le.transform(y)
Xtr, Xte, ytr, yte = train_test_split(X, ye, test_size=0.2, random_state=42, stratify=ye)
vals, cnt = np.unique(ytr, return_counts=True)
w = {v: len(ytr)/(len(vals)*n) for v, n in zip(vals, cnt)}
sw = np.array([w[v] for v in ytr])


### Train


In [ ]:
model = XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, min_child_weight=2,
    objective='multi:softprob', eval_metric='mlogloss', random_state=42, n_jobs=4)
model.fit(Xtr, ytr, sample_weight=sw)


### Evaluate — should clearly beat the majority-class baseline


In [ ]:
names = list(le.classes_)
pred = model.predict(Xte)
print(classification_report(yte, pred, target_names=names, digits=3))
print('confusion (rows=true):', names)
print(confusion_matrix(yte, pred))
print('baseline acc:', cnt.max()/cnt.sum())


### SHAP feature importance


In [ ]:
import shap
expl = shap.TreeExplainer(model)
sv = expl.shap_values(Xte[:200])
shap.summary_plot(sv, Xte[:200], feature_names=MODEL_FEATURES, plot_type='bar', show=True)


### Export (model + class-name mapping so the engine maps proba columns by name)


In [ ]:
joblib.dump({'model': model, 'classes': names, 'features': MODEL_FEATURES},
            '../backend/models/vetting_xgb.joblib')
print('saved -> backend/models/vetting_xgb.joblib', names)
